# Bonus Tasks: SQL with DuckDB

We repeat the Task 1 cleaning in SQL with DuckDB, in 4 layers: raw, cleansed, typed and staging.

The validation is not repeated here. The fixes applied in the cleansed layer come from the findings in Task 1 of `assessment.ipynb`.

## Setup

To run on Google Colab: choose `Runtime > Run all`

The notebook also runs from the `/analysis` folder of the repo, where it reads the files from `/data/raw`.

In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Pull the repo, so the file structure is the same as the local one
    if not os.path.exists("/content/flo-energy-data-analytics"):
        !git clone -q https://github.com/xanatower/flo-energy-data-analytics.git /content/flo-energy-data-analytics
    %cd /content/flo-energy-data-analytics

    # Install the packages listed in pyproject.toml into the Colab runtime
    !pip install -q uv
    !uv pip install -q --system -r pyproject.toml --group dev

    %cd analysis

In [2]:
from pathlib import Path
import duckdb

In [13]:
# specify file locations
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OFFER_STACKS_FILE = RAW_DIR / "DelayedOfferStacks_Energy_01-Jan-2023 to 31-Jan-2023.csv"
USEP_FILE = RAW_DIR / "USEP_Jan-2023.csv"

## 1. Spin up the database

In [ ]:
con = duckdb.connect()

## 2a. Load the original files

[Action] Raw layer

Both files are loaded as they are, with every column as text. This way a bad value such as `40 Jan 2023` is kept in the raw table, and the fix is applied in the cleansed layer.

The file path is passed to the SQL as a parameter.

In [5]:
OFFER_STACKS_RAW_SQL = """
    CREATE OR REPLACE TABLE offer_stacks_raw AS
    SELECT *
    FROM read_csv($file, skip = 2, header = true, all_varchar = true) -- skip = 2: the file has a title row and a blank row above the header
"""

USEP_RAW_SQL = """
    CREATE OR REPLACE TABLE usep_raw AS
    SELECT *
    FROM read_csv($file, header = true, all_varchar = true)
"""

con.execute(OFFER_STACKS_RAW_SQL, {"file": str(OFFER_STACKS_FILE)})
con.execute(USEP_RAW_SQL, {"file": str(USEP_FILE)});

## 2b. Clean the tables in SQL

[Action] Cleansed layer

Apply the 2 fixes from Task 1 to USEP. The columns stay as text with their original names, so this layer only shows the fixes:

- **Finding 1:** the invalid date `40 Jan 2023` on period 26 is corrected to 21 Jan 2023.
- **Finding 2:** the row with the invalid period `99` is removed. Its slot, 7 Jan period 27, stays as a gap.

Offer Stacks needs no fix, so it is carried over as it is.

In Task 1 the data is typed first and cleansed after. Here the order is the other way around, as `40 Jan 2023` can't be cast to a date until it is corrected.

In [ ]:
# No fix required
OFFER_STACKS_CLEANSED_SQL = """
    CREATE OR REPLACE TABLE offer_stacks_cleansed AS
    SELECT *
    FROM offer_stacks_raw
"""

USEP_CLEANSED_SQL = """
    CREATE OR REPLACE TABLE usep_cleansed AS
    SELECT
        "INFORMATION TYPE",
        -- Finding 1: the invalid date belongs to the missing 21 Jan period 26
        CASE WHEN "DATE" = $invalid_date THEN $corrected_date ELSE "DATE" END AS "DATE",
        "PERIOD",
        "USEP ($/MWh)",
        "LCP ($/MWh)",
        "DEMAND (MW)",
        "TCL (MW)"
    FROM usep_raw
    -- Finding 2: remove the row with the invalid period
    WHERE "PERIOD" <> $invalid_period
"""

con.execute(OFFER_STACKS_CLEANSED_SQL)

con.execute(
    USEP_CLEANSED_SQL,
    {
        "invalid_date": "40 Jan 2023",
        "corrected_date": "21 Jan 2023",
        "invalid_period": "99",
    },
);

[Action] Typed layer

Rename the columns to lower-case snake_case with no unit, and cast them to the right data type.

`CAST` is used and not `TRY_CAST`, so a bad value we haven't handled stops the load with an error, and doesn't turn into a NULL without us knowing.

In [7]:
OFFER_STACKS_TYPED_SQL = """
    CREATE OR REPLACE TABLE offer_stacks_typed AS
    SELECT
        CAST(strptime("Date", '%d-%b-%Y') AS DATE) AS "date",
        CAST("Period" AS INTEGER) AS "period",
        CAST("Lowest to Highest Offer Price ($/MWh)" AS DOUBLE) AS "lowest_to_highest_offer_price",
        CAST("Total Offer Capacity At Specified Offer Price (MW)" AS DOUBLE) AS "total_offer_capacity_at_specified_offer_price"
    FROM offer_stacks_cleansed
"""

USEP_TYPED_SQL = """
    CREATE OR REPLACE TABLE usep_typed AS
    SELECT
        "INFORMATION TYPE" AS "information_type",
        CAST(strptime("DATE", '%d %b %Y') AS DATE) AS "date",
        CAST("PERIOD" AS INTEGER) AS "period",
        CAST("USEP ($/MWh)" AS DOUBLE) AS "usep",
        CAST("LCP ($/MWh)" AS DOUBLE) AS "lcp",
        CAST("DEMAND (MW)" AS DOUBLE) AS "demand",
        CAST("TCL (MW)" AS DOUBLE) AS "tcl"
    FROM usep_cleansed
"""

con.execute(OFFER_STACKS_TYPED_SQL)
con.execute(USEP_TYPED_SQL);

[Action] Staging layer

Rename the offer columns to `bid_price` and `bid_volume`, and build the `datetime` from `date` and `period`:

`datetime = date + (period - 1) × 30 minutes`

Therefore, Period 1 corresponds to 00:00, Period 2 to 00:30, and so on.

In [8]:
OFFER_STACKS_STAGING_SQL = """
    CREATE OR REPLACE TABLE offer_stacks_staging AS
    SELECT
        CAST("date" AS TIMESTAMP) + ("period" - 1) * INTERVAL 30 MINUTE AS "datetime",
        "date",
        "period",
        "lowest_to_highest_offer_price" AS "bid_price",
        "total_offer_capacity_at_specified_offer_price" AS "bid_volume"
    FROM offer_stacks_typed
"""

USEP_STAGING_SQL = """
    CREATE OR REPLACE TABLE usep_staging AS
    SELECT
        CAST("date" AS TIMESTAMP) + ("period" - 1) * INTERVAL 30 MINUTE AS "datetime",
        "information_type",
        "date",
        "period",
        "usep",
        "lcp",
        "demand",
        "tcl"
    FROM usep_typed
"""

con.execute(OFFER_STACKS_STAGING_SQL)
con.execute(USEP_STAGING_SQL);

## Check the staging tables

[Analysis]

A small check that the SQL layers give the same result as Task 1:

- Offer Stacks should have 206,577 rows, covering all 1,488 periods of the month.
- USEP should have 1,487 rows, 1 period short, as the row with period `99` was removed.

In [9]:
STAGING_CHECK_SQL = """
    SELECT
        'offer_stacks_staging' AS "table_name",
        count(*) AS "row_count",
        count(DISTINCT "datetime") AS "periods",
        min("datetime") AS "first_period",
        max("datetime") AS "last_period"
    FROM offer_stacks_staging

    UNION ALL

    SELECT
        'usep_staging' AS "table_name",
        count(*) AS "row_count",
        count(DISTINCT "datetime") AS "periods",
        min("datetime") AS "first_period",
        max("datetime") AS "last_period"
    FROM usep_staging
"""

con.sql(STAGING_CHECK_SQL).df()

,table_name,row_count,periods,first_period,last_period
0,offer_stacks_staging,206577,1488,2023-01-01,2023-01-31 23:30:00
1,usep_staging,1487,1487,2023-01-01,2023-01-31 23:30:00


The only period missing from USEP should be 7 Jan period 27, the gap left by the removed row.

In [10]:
MISSING_PERIOD_SQL = """
    SELECT DISTINCT "datetime", "date", "period"
    FROM offer_stacks_staging
    WHERE "datetime" NOT IN (SELECT "datetime" FROM usep_staging)
"""

con.sql(MISSING_PERIOD_SQL).df()

,datetime,date,period
0,2023-01-07 13:00:00,2023-01-07,27


Quick inspection on the columns and data types of the USEP staging table

In [11]:
con.sql("DESCRIBE usep_staging").df()

,column_name,column_type,null,key,default,extra
0,datetime,TIMESTAMP,YES,None,None,None
1,information_type,VARCHAR,YES,None,None,None
2,date,DATE,YES,None,None,None
3,period,INTEGER,YES,None,None,None
4,usep,DOUBLE,YES,None,None,None
5,lcp,DOUBLE,YES,None,None,None
6,demand,DOUBLE,YES,None,None,None
7,tcl,DOUBLE,YES,None,None,None


Quick inspection on the columns and data types of the Offer Stacks staging table

In [12]:
con.sql("DESCRIBE offer_stacks_staging").df()

,column_name,column_type,null,key,default,extra
0,datetime,TIMESTAMP,YES,None,None,None
1,date,DATE,YES,None,None,None
2,period,INTEGER,YES,None,None,None
3,bid_price,DOUBLE,YES,None,None,None
4,bid_volume,DOUBLE,YES,None,None,None


**EOF**